In [1]:
!pip install transformers accelerate -q

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch, time

# Large model (verifier)
model = AutoModelForCausalLM.from_pretrained(
    "facebook/opt-1.3b", torch_dtype=torch.float16).to("cuda")

# Small model (drafter)
draft_model = AutoModelForCausalLM.from_pretrained(
    "facebook/opt-125m", torch_dtype=torch.float16).to("cuda")

tokenizer = AutoTokenizer.from_pretrained("facebook/opt-1.3b")
inputs = tokenizer("Explain what a KV cache is", return_tensors="pt").to("cuda")

# Without speculative decoding
start = time.time()
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=50)
baseline = time.time() - start
print(f"Baseline: {baseline:.2f}s")
print(tokenizer.decode(out[0], skip_special_tokens=True))

config.json:   0%|          | 0.00/653 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/2.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.63G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

Baseline: 2.71s
Explain what a KV cache is?
KV stands for KV-1S, a tank destroyer.                                  


In [3]:
# With speculative decoding
start = time.time()
with torch.no_grad():
    out_spec = model.generate(
        **inputs,
        max_new_tokens=50,
        assistant_model=draft_model
    )
spec_time = time.time() - start

print(f"Baseline:              {baseline:.2f}s")
print(f"Speculative decoding:  {spec_time:.2f}s")
print(f"Speedup:               {baseline/spec_time:.2f}x")
print(f"\nOutput: {tokenizer.decode(out_spec[0], skip_special_tokens=True)}")

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'min_new_tokens', 'use_cache'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Baseline:              2.71s
Speculative decoding:  1.55s
Speedup:               1.75x

Output: Explain what a KV cache is?
KV stands for KV-1S, a tank destroyer.                                  


In [4]:
results = []

for n_tokens in [20, 50, 100, 200]:
    # baseline
    start = time.time()
    with torch.no_grad():
        model.generate(**inputs, max_new_tokens=n_tokens)
    base = time.time() - start

    # speculative
    start = time.time()
    with torch.no_grad():
        model.generate(**inputs, max_new_tokens=n_tokens,
                      assistant_model=draft_model)
    spec = time.time() - start

    results.append((n_tokens, base, spec, base/spec))
    print(f"tokens={n_tokens}: baseline={base:.2f}s | spec={spec:.2f}s | speedup={base/spec:.2f}x")

tokens=20: baseline=2.26s | spec=1.83s | speedup=1.24x
tokens=50: baseline=1.40s | spec=1.52s | speedup=0.92x
tokens=100: baseline=1.75s | spec=1.47s | speedup=1.19x
tokens=200: baseline=3.70s | spec=2.77s | speedup=1.34x
